# 03 · Pruning: removing what the network does not need

**Goals of this notebook**

1. Understand why trained networks contain many unimportant parts.
2. **Edge pruning**: set individual weights (the edges of the network graph) to zero, and see why that alone does *not* make the model smaller or faster on the Arduino.
3. **Node pruning**: remove whole neurons (the nodes), so the network *really* gets smaller.
4. Learn why **fine-tuning** after pruning is essential, and compare pruning with simply training a smaller network.
5. Learn the practical side: pruning schedules, choosing which layers to prune, compression, measured speed, and combining pruning with quantization.

In [ ]:
import copy

import numpy as np
import torch
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# EXTERNAL (shared with the other notebooks, code shown in notebook 01):
#   load_gesture_data  the synthetic sensor data (see the comment block in notebook 01)
#   MLP, count_params  the small network class and a parameter counter
#   train, accuracy, set_seed   the training loop, test accuracy, repeatable random numbers
# Why external: these are used in every notebook. Notebook 01 explains them line by
# line; repeating them here would only copy code. Everything about pruning, which is
# the topic of THIS notebook, is written out below.
# ---------------------------------------------------------------------------
import sys, pathlib
# Only needed so Python can find the course folder `aiedge` (the helpers imported below).
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from aiedge.data import load_gesture_data
from aiedge.models import MLP, count_params
from aiedge.training import train, accuracy, set_seed

data = load_gesture_data()
set_seed(0)
baseline = MLP(hidden=(64, 32))
train(baseline, data, augment=True, verbose=False)
base_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline: {baseline}, test accuracy {base_acc:.1%}, {count_params(baseline):,} parameters")

## 1. Why can we remove parts of a trained network?

Networks are usually trained **larger than necessary**. That makes training easier: with more weights there are more ways to reach a good solution. But once trained, many weights end up close to zero and many neurons barely contribute. Your brain does something similar: a toddler has far more connections between neurons than an adult. Unused connections are removed during childhood (*synaptic pruning*).

In a famous 2015 paper, Han et al. removed **~90% of the weights** of the image network AlexNet without losing accuracy. Later work (the *lottery ticket hypothesis*, Frankle & Carbin, 2019) even showed that big networks contain small sub-networks that could have been trained on their own.

Remember the network as a graph: **neurons are nodes, weights are edges**. We can prune either:

| | **Edge pruning** (unstructured) | **Node pruning** (structured) |
|---|---|---|
| removes | individual weights | whole neurons (with all their weights) |
| result | same matrices, many zeros | smaller matrices |
| flexibility | very high: any weight can go | lower: all or nothing per neuron |
| gain on a microcontroller | only with special sparse code | immediately: fewer MACs, less memory |

## 2. Edge pruning

### The criterion: weight magnitude
Which weights are unimportant? The simplest and very effective rule: **the ones closest to zero**. A weight of 0.001 multiplies its input by almost nothing, so setting it to exactly 0 changes the output very little.

We describe a pruned network with a **mask**: a 0/1 matrix of the same shape as each weight matrix. 1 = keep, 0 = pruned. The **sparsity** is the fraction of weights that are zero.

Let's write the pruning ourselves: find the threshold below which a fraction `sparsity` of all weights lies, and zero everything below it.

In [ ]:
def sparsity_of(model):
    """Fraction of weights (not biases) that are exactly zero."""
    ws = [p for n, p in model.named_parameters() if n.endswith("weight")]
    return sum(int((w == 0).sum()) for w in ws) / sum(w.numel() for w in ws)

def count_nonzero_params(model):
    return sum(int((p != 0).sum()) for p in model.parameters())

def prune_smallest(model, sparsity):
    """Edge pruning: zero the `sparsity` fraction of weights with the smallest |w| (all layers together)."""
    m = copy.deepcopy(model)
    all_w = torch.cat([l.weight.detach().abs().flatten() for l in m.layers])
    threshold = torch.quantile(all_w, sparsity)
    masks = {}
    with torch.no_grad():
        for i, l in enumerate(m.layers):
            mask = (l.weight.abs() > threshold).float()
            l.weight.mul_(mask)
            masks[f"layers.{i}.weight"] = mask
    return m, masks

pruned, masks = prune_smallest(baseline, 0.5)
print(f"sparsity: {sparsity_of(pruned):.1%}   non-zero parameters: {count_nonzero_params(pruned):,} / {count_params(pruned):,}")
print(f"test accuracy after removing half of all weights: {accuracy(pruned, data.X_test, data.y_test):.1%}  (baseline {base_acc:.1%})")

Half of the weights are gone, and the accuracy hardly changed! Let's go further. We prune increasingly more, **in one shot**, without any retraining:

In [ ]:
sparsities = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]
oneshot = []
for sp in sparsities:
    m, _ = prune_smallest(baseline, sp)
    oneshot.append(accuracy(m, data.X_test, data.y_test))
    print(f"sparsity {sp:4.0%}: test accuracy {oneshot[-1]:.1%}")

### Fine-tuning after pruning
Beyond ~80% sparsity the accuracy collapses: we are now removing weights that matter. But the remaining weights can **compensate** if we train a little more. This is **fine-tuning**.

One crucial detail: during fine-tuning the optimizer would happily give the pruned weights non-zero values again, and the pruned weights would "grow back". So after **every** optimizer step we multiply the weights by their mask again. (`train(..., masks=masks)` does this. Forgetting it is a classic bug: you think you deployed a pruned model, but it isn't sparse any more!)

In [ ]:
finetuned = []
for sp in sparsities:
    m, masks = prune_smallest(baseline, sp)
    train(m, data, masks=masks, lr=1e-3, epochs=40, patience=10, augment=True, verbose=False)
    finetuned.append(accuracy(m, data.X_test, data.y_test))
    print(f"sparsity {sp:4.0%}: one-shot {oneshot[sparsities.index(sp)]:.1%}  ->  fine-tuned {finetuned[-1]:.1%}"
          f"   (check: sparsity after fine-tuning = {sparsity_of(m):.1%})")

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(sparsities, oneshot, "o-", label="one-shot (no fine-tuning)")
ax.plot(sparsities, finetuned, "s--", label="pruned + fine-tuned")
ax.axhline(base_acc, color="gray", ls=":", lw=1.5)
ax.set_xlabel("sparsity (fraction of weights removed)"); ax.set_ylabel("test accuracy")
ax.set_title("Edge pruning"); ax.legend(); plt.show()

With fine-tuning we can remove **~90% of the weights** with little or no loss. At 98%, however, even fine-tuning cannot repair the damage of one big cut. That is where **gradual pruning** helps.

### Gradual pruning with a schedule
Instead of cutting everything at once: prune a little, fine-tune, prune a bit more, and so on. The **pruning schedule** says how sparse the network should be at each step. A popular choice is **polynomial decay** (TensorFlow's Model Optimization Toolkit calls it `PolynomialDecay`). It starts at an initial sparsity and approaches the final one, with big steps at first, while there are still plenty of unimportant weights, and small steps at the end, when every weight counts:

$$\text{sparsity}(t) = s_{final} + (s_{initial} - s_{final})\left(1 - \frac{t - t_{begin}}{t_{end} - t_{begin}}\right)^{3}$$

In TensorFlow, $t$ counts optimizer steps (mini-batches) and the mask is updated every few steps during normal training. We keep it simpler: 5 stages with 8 epochs of fine-tuning each, so 40 epochs in total, the same budget as the one-shot fine-tuning above.

In [ ]:
def polynomial_sparsity(step, initial, final, begin, end, power=3):
    """Target sparsity at `step`: from `initial` at `begin` to `final` at `end`, big steps first."""
    progress = min(max((step - begin) / (end - begin), 0.0), 1.0)
    return final + (initial - final) * (1 - progress) ** power

FINAL = 0.98
gradual = copy.deepcopy(baseline)
for stage in range(5):
    target = polynomial_sparsity(stage, initial=0.5, final=FINAL, begin=0, end=4)
    gradual, masks = prune_smallest(gradual, target)    # weights pruned earlier are 0, so they stay pruned
    train(gradual, data, masks=masks, lr=1e-3, epochs=8, patience=8, augment=True, verbose=False)
    print(f"stage {stage}: sparsity {sparsity_of(gradual):5.1%}, test accuracy {accuracy(gradual, data.X_test, data.y_test):.1%}")

print(f"\n{FINAL:.0%} sparsity: gradual {accuracy(gradual, data.X_test, data.y_test):.1%}"
      f"  vs  one-shot + fine-tuning {finetuned[sparsities.index(FINAL)]:.1%}")

At this extreme sparsity the gradual schedule wins clearly: after every cut, the network gets time to shift the work to the remaining weights before the next cut. Up to about 90% sparsity, one cut plus fine-tuning is usually good enough.

### Where were the weights removed?
We used **one global threshold** for all layers. Let's see what fraction was removed in each layer at 90% sparsity:

In [ ]:
m90, masks90 = prune_smallest(baseline, 0.9)
for i, l in enumerate(m90.layers):
    print(f"layer {i} ({l.in_features:3d} -> {l.out_features:2d}): {sparsity_of(torch.nn.ModuleList([l])):5.1%} of its weights removed")

The first layer (300 inputs, 19 200 weights) loses the most: many of the 300 inputs are redundant (neighbouring readings in time are similar). The small last layer is almost untouched: every one of its weights matters. A global threshold finds this automatically. (Pruning every layer by the same fraction instead is one of the exercises.)

### Selective pruning: which layers can take it?
Here the global threshold sorted it out automatically. Often, though, you decide yourself which layers to prune and how much: **selective pruning**. The usual first step is to measure each layer's **sensitivity**: prune one layer at a time, leave the others alone, and see how quickly the accuracy drops.

In [ ]:
@torch.no_grad()
def prune_one_layer(model, layer_index, sparsity):
    """Edge pruning of one layer only, with that layer's own threshold."""
    m = copy.deepcopy(model)
    w = m.layers[layer_index].weight
    threshold = torch.quantile(w.abs().flatten(), sparsity)
    w.mul_((w.abs() > threshold).float())
    return m

test_sparsities = [0.5, 0.8, 0.9, 0.95]
print("only one layer pruned, no fine-tuning")
print(f"{'sparsity of that layer:':<42}" + "".join(f"{sp:>8.0%}" for sp in test_sparsities))
for i, layer in enumerate(baseline.layers):
    accs = [accuracy(prune_one_layer(baseline, i, sp), data.X_test, data.y_test) for sp in test_sparsities]
    label = f"layer {i} ({layer.in_features:3d} -> {layer.out_features:2d}, {layer.weight.numel():6,} weights)"
    print(f"{label:<42}" + "".join(f"{a:>8.1%}" for a in accs))

The small output layer (only 160 weights) is the most sensitive: each of its weights connects a hidden neuron directly to a gesture score. The big first layer tolerates much more. A selective plan therefore prunes the first layer hard, the middle one moderately, and the output layer little or not at all. In TensorFlow you do this by wrapping only the chosen layers with `prune_low_magnitude`.

There is a second reason to leave small layers alone: sparse code only gets faster for big, very sparse layers (we will measure this below).

### The catch: zeros still cost memory and time
Our C code on the Arduino stores every weight matrix as a dense block of numbers and multiplies **every** weight, including the zeros. So a 90%-sparse model is exactly as big and as slow as the original!

To profit, you need a **sparse format** that only stores the non-zero weights, plus their positions. A common one is **CSR** (compressed sparse row): for every non-zero weight you store its value *and its column index*, plus one pointer per row. With int8 values and 16-bit (2-byte) indices, every surviving weight costs 3 bytes instead of 1:

In [ ]:
def sparse_storage_bytes(model, bytes_per_value=1, bytes_per_index=2):
    """Bytes for the weights in CSR format: value + column index per non-zero weight, one pointer per row."""
    total = 0
    for layer in model.layers:
        nnz = int((layer.weight != 0).sum())
        total += nnz * (bytes_per_value + bytes_per_index) + (layer.weight.shape[0] + 1) * bytes_per_index
        total += layer.bias.numel() * 4                   # biases stay dense, 32-bit
    return total

def dense_int8_bytes(model):
    return sum(l.weight.numel() + 4 * l.bias.numel() for l in model.layers)   # int8 weights, int32 biases

print(f"{'sparsity':>9} {'dense int8 bytes':>17} {'CSR int8 bytes':>15}")
for sp in [0.0, 0.5, 0.67, 0.8, 0.9, 0.95]:
    m, _ = prune_smallest(baseline, sp)
    print(f"{sp:>9.0%} {dense_int8_bytes(m):>17,} {sparse_storage_bytes(m, 1, 2):>15,}")

CSR only pays off above ~67% sparsity (because each kept weight costs 3 bytes instead of 1). On top of that, sparse multiplication jumps around in memory and needs extra index arithmetic per MAC, so on a small CPU it only gets *faster* at high sparsity. On big hardware such as GPUs, special sparse patterns are needed (e.g. "2 out of every 4 weights are zero").

### Simply zip it?
Another way to see the effect of pruning is to **compress** the weights with a general-purpose algorithm, the one used in `.zip` files. Long runs of zeros compress very well. Python's built-in `zlib` module uses the same algorithm. (Int8 quantization keeps a zero weight exactly zero, so we can zip the int8 weights.)

In [ ]:
import zlib   # part of Python: the "deflate" compression used in .zip files

def int8_weight_bytes(model):
    """All weights as int8 numbers (symmetric, one scale per neuron), as one block of bytes."""
    blob = b""
    for layer in model.layers:
        w = layer.weight.detach()
        s = w.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127
        blob += torch.round(w / s).to(torch.int8).numpy().tobytes()
    return blob

print(f"{'sparsity':>9} {'int8 weights':>13} {'zipped':>8} {'CSR':>8}   (bytes, weights only)")
for sp in [0.0, 0.5, 0.8, 0.9, 0.95]:
    m, _ = prune_smallest(baseline, sp)
    blob = int8_weight_bytes(m)
    csr = sparse_storage_bytes(m, 1, 2) - 4 * sum(l.bias.numel() for l in m.layers)   # without the biases
    print(f"{sp:>9.0%} {len(blob):>13,} {len(zlib.compress(blob, 9)):>8,} {csr:>8,}")

The zipped size drops quickly with sparsity, even below CSR. So why not zip the model for the board?

Zipping only helps with **storing and sending** a model: a smaller app, a faster download, a quicker over-the-air update. To *run* the model, it must be unzipped first, into RAM, and our board has 4x less RAM than flash. After unzipping, the network still multiplies every zero. So the zipped size tells you how much information is left in the weights, not what the model costs to run on a microcontroller.

### Is a sparse model faster? Let's measure
In theory, sparse multiplication only pays off at high sparsity. Let's measure it on your laptop with PyTorch's CSR matrices: one prediction of our network (dense vs CSR), and one big layer of 2000 × 2000 weights. Timing is noisy because other programs run too, so we take the fastest of several runs. Your numbers will differ from your neighbour's, but the pattern should be the same.

In [ ]:
import time, warnings
# PyTorch prints a warning that its CSR support is still "beta"; for this small test that is fine.
warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta")

def time_us(f, repeats=5, n=300):
    """Microseconds per call of f(): the fastest of `repeats` runs of n calls (the least disturbed one)."""
    f()                                                   # warm-up
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        for _ in range(n):
            f()
        best = min(best, (time.perf_counter() - start) / n)
    return best * 1e6

def forward_with(weights, biases, x):
    """One prediction with plain matrix-vector products; the weights may be dense or sparse."""
    for i, (w, b) in enumerate(zip(weights, biases)):
        x = w @ x + b
        if i < len(weights) - 1:
            x = torch.relu(x)
    return x

x = data.X_test[0]
biases = [l.bias.detach() for l in baseline.layers]
print("our network, one prediction:")
print(f"{'sparsity':>9} {'dense [us]':>11} {'CSR [us]':>9}")
with torch.no_grad():
    for sp in [0.0, 0.5, 0.9, 0.95]:
        m, _ = prune_smallest(baseline, sp)
        dense = [l.weight.detach().clone() for l in m.layers]
        sparse = [w.to_sparse_csr() for w in dense]
        print(f"{sp:>9.0%} {time_us(lambda: forward_with(dense, biases, x)):>11.1f} "
              f"{time_us(lambda: forward_with(sparse, biases, x)):>9.1f}")

print("\none big layer, 2000 x 2000 weights:")
print(f"{'sparsity':>9} {'dense [us]':>11} {'CSR [us]':>9}")
torch.manual_seed(0)
v = torch.randn(2000)
for sp in [0.0, 0.5, 0.9, 0.95, 0.99]:
    W = torch.randn(2000, 2000)
    W[torch.rand(2000, 2000) < sp] = 0.0
    W_csr = W.to_sparse_csr()
    print(f"{sp:>9.0%} {time_us(lambda: W @ v, n=50):>11.1f} {time_us(lambda: W_csr @ v, n=50):>9.1f}")

What you should see:
* For **our small network**, the sparse version is **slower at every sparsity**. The bookkeeping (reading indices, jumping around in memory) costs more than the skipped MACs save. A pruned model can really run *slower* than the original: TensorFlow's pruning examples show the same.
* For the **big layer**, sparse only wins at very high sparsity, around 90% or more on most laptops.

So edge pruning only brings speed with large layers, very high sparsity and special sparse code. Common microcontroller libraries such as TensorFlow Lite for Microcontrollers store and multiply the weights densely, so they don't profit from the zeros either.

**Edge pruning is powerful, but its gains are only real if your software and hardware can exploit sparsity.** That brings us to node pruning.

## 3. Node pruning

If an entire neuron is removed, the matrices simply become smaller. That works with any ordinary dense code, ours included.

### Which neurons are unimportant?
A neuron matters if (1) its output is often large, **and** (2) the next layer actually listens to it (large outgoing weights). We combine both into one importance score:

$$\text{importance}_j = \underbrace{\text{mean}\big(|a_j|\big)}_{\text{average output on real data}} \times \underbrace{\|W_{\text{next}}[:, j]\|}_{\text{size of its outgoing weights}}$$

In [ ]:
@torch.no_grad()
def neuron_importance(model, X):
    """Importance of every hidden neuron: one tensor per hidden layer."""
    model.eval()
    scores, x = [], X
    for i, layer in enumerate(model.layers[:-1]):         # all layers except the output layer
        x = torch.relu(layer(x))                          # the outputs a_j of this hidden layer
        mean_act = x.abs().mean(dim=0)                    # mean |a_j| over the data, one number per neuron
        out_w = model.layers[i + 1].weight.norm(dim=0)    # size of column j of the NEXT layer's weights
        scores.append(mean_act * out_w)
    return scores

In [ ]:
scores = neuron_importance(baseline, data.X_train)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for k, (ax, s) in enumerate(zip(axes, scores)):
    ax.bar(range(len(s)), s.sort(descending=True).values.numpy(), color="C0")
    ax.set_title(f"hidden layer {k + 1}: neuron importance (sorted)")
    ax.set_xlabel("neuron (sorted by importance)"); ax.set_ylabel("importance")
fig.tight_layout(); plt.show()

A handful of neurons do most of the work; many contribute little. Let's check that the score makes sense: remove the **single least** important neuron of hidden layer 1, and then (separately) the **single most** important one, and see how much the network's output changes.

### Network surgery
Removing neuron $j$ from hidden layer $l$ means:
* delete **row $j$** of the weight matrix of layer $l$ and **entry $j$** of its bias (the neuron's incoming connections);
* delete **column $j$** of the weight matrix of layer $l+1$ (its outgoing connections).

In PyTorch this is just indexing: `W_l[keep]`, `b_l[keep]`, `W_next[:, keep]`. The function `remove_neurons` below does it for all layers.

In [ ]:
@torch.no_grad()
def remove_neurons(model, keep):
    """Build a smaller MLP that keeps only the neurons listed in `keep`.

    keep[l] = indices of the neurons to keep in hidden layer l.
    """
    keep = [torch.as_tensor(k).sort().values for k in keep]
    small = MLP(hidden=[len(k) for k in keep], in_features=model.sizes[0], n_classes=model.sizes[-1])
    prev_keep = torch.arange(model.sizes[0])             # all inputs are kept
    for l, (old, new) in enumerate(zip(model.layers, small.layers)):
        # rows of this layer = this layer's neurons that we keep (all outputs for the last layer)
        rows = keep[l] if l < len(keep) else torch.arange(old.out_features)
        # take those rows, and only the columns that point at neurons kept in the layer before
        new.weight.copy_(old.weight[rows][:, prev_keep])
        new.bias.copy_(old.bias[rows])
        prev_keep = rows
    return small

In [ ]:
order = scores[0].argsort()                      # least important first
all_l2 = torch.arange(32)
with torch.no_grad():
    ref = baseline(data.X_test)
    for name, drop in [("least important", order[0]), ("most important", order[-1])]:
        keep_l1 = torch.tensor([j for j in range(64) if j != drop])
        smaller = remove_neurons(baseline, [keep_l1, all_l2])
        change = (smaller(data.X_test) - ref).abs().mean()
        print(f"remove the {name:15s} neuron (#{int(drop):2d}) -> {smaller}: "
              f"mean logit change {change:.4f}, accuracy {accuracy(smaller, data.X_test, data.y_test):.1%}")

Removing the most important neuron changes the outputs several times more than removing the least important one, so the score does what it should. One neuron out of 64 barely changes the *accuracy* either way: the network is robust. But the damage adds up when we remove many neurons, as the next experiment shows.

### Pruning many neurons, with and without fine-tuning
Now we shrink both hidden layers to a range of sizes. For every size we compare three things:
1. **one-shot**: keep the most important neurons, no retraining;
2. **pruned + fine-tuned**: the same, then train a bit more;
3. **trained from scratch**: a brand-new network of the small size, trained normally.

The third one is an important *control experiment*: if it is just as good, then pruning did not help, and we could simply have trained a small network!

### Putting the steps together
`prune_neurons` scores the neurons, keeps the best `new_hidden[l]` of every hidden layer, and does the surgery. `macs` counts the multiply-accumulate operations of one prediction: one per weight.

In [ ]:
def prune_neurons(model, X, new_hidden):
    """Keep the `new_hidden[l]` most important neurons of every hidden layer."""
    scores = neuron_importance(model, X)
    keep = [s.argsort(descending=True)[:n] for s, n in zip(scores, new_hidden)]
    return remove_neurons(model, keep)

def macs(model):
    return sum(l.in_features * l.out_features for l in model.layers)

In [ ]:
sizes = [(64, 32), (48, 24), (32, 16), (24, 12), (16, 8), (8, 4)]
res = {"one-shot": [], "pruned + fine-tuned": [], "trained from scratch": []}
params = []
for h in sizes:
    small = prune_neurons(baseline, data.X_train, h)
    res["one-shot"].append(accuracy(small, data.X_test, data.y_test))
    train(small, data, lr=1e-3, epochs=60, patience=15, augment=True, verbose=False)
    res["pruned + fine-tuned"].append(accuracy(small, data.X_test, data.y_test))
    set_seed(0)
    scratch = MLP(hidden=h)
    train(scratch, data, augment=True, verbose=False)
    res["trained from scratch"].append(accuracy(scratch, data.X_test, data.y_test))
    params.append(count_params(small))
    print(f"{str(h):9s} {params[-1]:6,} params ({macs(small):6,} MACs): one-shot {res['one-shot'][-1]:.1%}, "
          f"fine-tuned {res['pruned + fine-tuned'][-1]:.1%}, from scratch {res['trained from scratch'][-1]:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
for (name, accs), style in zip(res.items(), ["o-", "s--", "^:"]):
    ax.plot(params, accs, style, label=name)
ax.axhline(base_acc, color="gray", ls=":", lw=1.5)
ax.set_xscale("log"); ax.set_xlabel("parameters (log scale)"); ax.set_ylabel("test accuracy")
ax.set_title("Node pruning"); ax.legend(); plt.show()

**Interpreting the results**
* **One-shot** node pruning hurts a lot: removing a neuron removes *all* its connections at once.
* **Fine-tuning** recovers most of it: down to roughly a quarter of the parameters, the pruned network is about as good as the baseline.
* **From scratch** is often just as good for a network this small! Researchers found the same for many models (Liu et al., *Rethinking the Value of Network Pruning*, 2019). Pruning helps most for big networks, when training a small one from scratch is hard. It also helps when you start from a network trained on lots of data or with a teacher (notebook 04): pruning keeps that knowledge, while a fresh network has to learn everything again.

That is a useful lesson in general: **always compare against a simple baseline** before concluding a fancy method works.

## 4. Edge vs node pruning on the Arduino

What matters for the board: accuracy versus actual bytes (int8 weights). For edge pruning we count the CSR format; for node pruning the plain dense format.

In [ ]:
edge_bytes = []
for sp in sparsities:
    m, _ = prune_smallest(baseline, sp)
    edge_bytes.append(min(sparse_storage_bytes(m, 1, 2), dense_int8_bytes(m)))   # pick the smaller format
node_bytes = [dense_int8_bytes(MLP(hidden=h)) for h in sizes]

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.array(edge_bytes) / 1024, finetuned, color="C0", marker="o", label="edge pruning (CSR)")
ax.plot(np.array(node_bytes) / 1024, res["pruned + fine-tuned"], color="C1", ls="--", marker="s",
        label="node pruning (dense)")
ax.set_xscale("log"); ax.set_xlabel("int8 model size [KB] (log scale)"); ax.set_ylabel("test accuracy")
ax.set_title("Accuracy vs memory after fine-tuning"); ax.legend(); plt.show()

Both reach similar sizes for similar accuracy. Node pruning has the big practical advantage that it needs **no special code**: the smaller network runs on our existing C code and is faster in proportion to its MACs. That's why notebook 05 uses **node pruning** for the deployed model.

### Edge pruning + quantization
Pruning and quantization combine well. This is the classic recipe of *Deep Compression* (Han, Mao & Dally, 2016): prune, fine-tune, then quantize. Let's prune to 90%, fine-tune, and quantize to int8 with post-training quantization from notebook 02. Two things to check: does the accuracy hold, and do the pruned weights stay zero?

In [ ]:
# ---------------------------------------------------------------------------
# EXTERNAL (shared, the code is shown and explained in notebook 02):
#   QuantMLP   wraps a trained MLP and simulates int8 inference: call calibrate(X), then use it like a model
# ---------------------------------------------------------------------------
from aiedge.quantization import QuantMLP

edge90, masks = prune_smallest(baseline, 0.9)
train(edge90, data, masks=masks, lr=1e-3, epochs=40, patience=10, augment=True, verbose=False)
q90 = QuantMLP(copy.deepcopy(edge90))
q90.calibrate(data.X_train)

with torch.no_grad():
    w_int8 = [torch.round(l.weight / q90.weight_scale(l)[:, None]) for l in q90.mlp.layers]   # the int8 weights
zeros = sum(int((w == 0).sum()) for w in w_int8) / sum(w.numel() for w in w_int8)
print(f"pruned 90% + fine-tuned (float32): {accuracy(edge90, data.X_test, data.y_test):.1%}")
print(f"... then int8 PTQ:                 {accuracy(q90, data.X_test, data.y_test):.1%}   "
      f"zero weights after quantization: {zeros:.1%}")
print(f"memory: dense int8 {dense_int8_bytes(edge90):,} B, CSR int8 {sparse_storage_bytes(edge90, 1, 2):,} B "
      f"(float32 baseline {4 * count_params(baseline):,} B)")

The accuracy holds, and the zeros stay zeros: weights use zero-point 0, so $\text{round}(0 / s) = 0$. The savings multiply: int8 makes every stored weight 4x smaller, and the sparse format stores only 10% of them. Notebook 05 uses the same order, pruning first and quantization last, but with node pruning.

## Summary

* Trained networks are over-parameterised; many weights and neurons can be removed.
* **Edge pruning** (smallest |w| → 0): up to ~90% sparsity with fine-tuning. But zeros only save memory/time with sparse formats and sparse kernels.
* **Node pruning** (remove whole neurons, by importance): matrices really shrink; works with any dense code.
* Always **fine-tune** after pruning, and keep the mask during fine-tuning.
* Always compare with **training the small model from scratch**.
* For very high sparsity, prune **gradually** with a schedule (for example polynomial decay).
* Layers differ in **sensitivity**: measure it and prune **selectively** (small output layers least).
* Zipping a sparse model shows how much information is left, but the board still has to run the full matrices. Sparse code is only faster for big, very sparse layers.
* Pruning and int8 quantization stack: pruned weights stay exactly zero after quantization.

## Exercises

1. ⭐ **Forget the mask.** Prune the baseline to 90% sparsity and fine-tune *without* `masks=`. Measure `sparsity_of(model)` after training. What happened, and why is that dangerous?
2. ⭐ **Global vs per-layer.** Change `prune_smallest` so that every layer gets its own threshold (`torch.quantile` of that layer's |w| only), then fine-tune with `masks=`. Compare with the global version. Which is better, and why?
3. **Gradual pruning.** Shrink the baseline from (64, 32) to (16, 8) in 4 steps (48/24, 32/16, 24/12, 16/8) with a short fine-tune (20 epochs) after each step. Is it better than one-shot pruning + fine-tuning?
4. **Another importance score.** Change the score to only the L1 norm of the *incoming* weights of each neuron (`model.layers[l].weight.abs().sum(dim=1)`) and use `remove_neurons` with it. Does the choice of score matter after fine-tuning?
5. **Think.** Our input layer has 300 inputs. Node pruning cannot remove inputs. Which inputs could you remove instead, and what would that mean physically (hint: channels and time steps)?